In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import os
import fs
import src.process_data as dp
import mdp_utils
from scipy import stats
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'src'

In [49]:
NUM_VALS_PER_FEATURE = [3, 3, 3]
NUM_FEATURES = len(NUM_VALS_PER_FEATURE)
NUM_USER_STATES = np.prod(NUM_VALS_PER_FEATURE)
NUM_CLUSTERS = 6
MAX_COUNT = 2
NUM_COUNT_STATES = (MAX_COUNT+1)**NUM_CLUSTERS
NUM_STATES = NUM_USER_STATES * NUM_COUNT_STATES 
NUM_ACTIONS = NUM_CLUSTERS

cluster_col = 'cluster_all'
state_features = ['tiredness', 'time_avail', 'pu_state']
feature_names = ['Tiredness', 'Time available', 'Perceived usefulness']
cluster_vars = ['obs_diff', 'obs_time', 'obs_liked', 'obs_pu']
cluster_cols = [col + '_cluster' for col in cluster_vars]

reward_cols = ["obs_time", "obs_liked", "obs_pu", "r_expert", "r_diversity"]
obj_names = ["Time Required", "Fun", "Perceived Usefulness", "Expert Usefulness", "Novelty"]

NUM_OBJECTIVES = len(reward_cols)

In [ ]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\functions\\3'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

# Load data
action_data = pd.read_csv(os.path.join(data_folder, 'challenge_info.csv'))
samples = pd.read_csv(os.path.join(data_folder, 'processed_samples.csv'))

actions_clustered, _, _ = dp.cluster_actions(action_data, cluster_vars, num_clusters=6)

df, initial_distribution = dp.process_samples(samples, actions_clustered, state_features, NUM_VALS_PER_FEATURE, cluster_col, MAX_COUNT, NUM_CLUSTERS)

In [51]:
df = df.merge(actions_clustered[['action_id'] + cluster_cols], on='action_id', how='left')

df.head()

,user_id,timestep,action_id,category_id,completed,obs_diff,obs_time,obs_liked,obs_pu,r_expert,...,s_count_next,s_count,r_diversity,c_idx,s_idx,sp_idx,obs_diff_cluster,obs_time_cluster,obs_liked_cluster,obs_pu_cluster
0,1,1,44,1,True,0.428571,0.285714,0.571429,0.714286,0.333333,...,"[0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0]",1.0,0,1,10,5,3,5,3
1,1,2,92,3,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,"[0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 1, 0]",1.0,3,10,13,3,3,4,1
2,1,3,101,3,True,0.857143,0.571429,1.000000,1.000000,0.250000,...,"[0, 0, 0, 1, 1, 0]","[0, 0, 0, 0, 1, 0]",1.0,3,13,14,3,0,4,5
3,1,4,46,1,True,0.571429,0.428571,0.571429,1.000000,0.250000,...,"[1, 0, 0, 1, 1, 0]","[0, 0, 0, 1, 1, 0]",1.0,12,14,5,2,4,3,5
4,1,5,79,3,False,0.000000,0.000000,0.000000,0.000000,0.000000,...,"[1, 0, 0, 1, 1, 0]","[1, 0, 0, 1, 1, 0]",1.0,255,5,14,3,3,3,3


In [52]:

R_probs_diff = mdp_utils.compute_reward_probabilities(df, 'obs_diff', NUM_USER_STATES, NUM_ACTIONS, 'obs_diff_cluster')
R_probs_fun = mdp_utils.compute_reward_probabilities(df, 'obs_liked', NUM_USER_STATES, NUM_ACTIONS, 'obs_liked_cluster')
R_probs_pu = mdp_utils.compute_reward_probabilities(df, 'obs_pu', NUM_USER_STATES, NUM_ACTIONS, 'obs_pu_cluster')
R_probs_time = mdp_utils.compute_reward_probabilities(df, 'obs_time', NUM_USER_STATES, NUM_ACTIONS, 'obs_time_cluster')

R_probs = {'obs_diff': R_probs_diff, 'obs_liked': R_probs_fun, 'obs_pu': R_probs_pu, 'obs_time': R_probs_time}